In [ ]:

# 2026-09-30 run against production
from csae_pyutils import gsheet_to_df
from tqdm import tqdm
from django.core.exceptions import ObjectDoesNotExist
from apis_core.apis_metainfo.models import Collection, Uri, Label
from apis_core.apis_entities.models import Work, Person
from apis_core.apis_vocabularies.models import LabelType, PersonWorkRelation, WorkWorkRelation
from apis_core.apis_relations.models import PersonWork, WorkWork

In [ ]:
fackel_col = Collection.objects.filter(name__startswith="Karl Kraus: Die Fackel").first()
col = Collection.objects.filter(name__startswith="Karl Kraus: SemanticKraus").first()
domain = "semantickraus"

In [ ]:
werk_link_label = LabelType.objects.get(id=1214)
bibl_label = LabelType.objects.get(id=1235)
person_work_rel_type = PersonWorkRelation.objects.get(name="hat geschaffen")
work_work_rel = WorkWorkRelation.objects.get(name="enthalten in")
die_fackel = Work.objects.get(id=41244)
werk_link_label, bibl_label, person_work_rel_type, work_work_rel, die_fackel



In [ ]:
df = gsheet_to_df("1TsRWWSWYhm0eRH-qpv705Xw6qUCakmHODWoFlqtVSVQ")

In [ ]:
author_cache = {}
for i, row in tqdm(df.iterrows(), total=len(df)):
    tmp_uri = row["uri"]
    pmb_uri, _ = Uri.objects.get_or_create(uri=tmp_uri, domain=domain)
    fackel_uri, _ = Uri.objects.get_or_create(uri=row["werk_link"], domain="fackel")
    entity = pmb_uri.entity
    if entity:
        pass
    else:
        item = {
            "name": row["title"][:254],
        }
        entity = Work.objects.create(**item)
    entity.start_date_written = row["date-combined"]
    entity.save()
    Label.objects.get_or_create(label=row["werk_link"][:245], label_type=werk_link_label, temp_entity=entity)
    Label.objects.get_or_create(label=row["bibliography"][:245], label_type=bibl_label, temp_entity=entity)
    entity.collection.add(col)
    entity.collection.add(fackel_col)
    pmb_uri.entity = entity
    fackel_uri.entity = entity
    fackel_uri.save()
    work = Work.objects.get(id=entity.id)
    WorkWork.objects.get_or_create(
        related_worka=work,
        related_workb=die_fackel,
        start_date_written=row["date-combined"],
        end_date_written=row["date-combined"],
        relation_type=work_work_rel
    )
    try:
        row["authors"].split("|")
        authors = True
    except AttributeError:
        authors = False
    if authors:
        for x in row["authors"].split("|"):
            author_id = f"https://fackel.oeaw.ac.at/?p={x}"
            if author_id in ["https://fackel.oeaw.ac.at/?p=fackelp57671","https://fackel.oeaw.ac.at/?p=fackelp57783", "https://fackel.oeaw.ac.at/?p=fackelp57964"]:
                continue
            
            try:
                author_entity = author_cache[author_id]
            except KeyError:
                temp_ent = Uri.objects.get(uri=author_id).entity
                author_ent = Person.objects.get(id=temp_ent.id)
                author_cache[author_id] = author_ent
                author_entity = author_cache[author_id]
            PersonWork.objects.get_or_create(
                    related_person=author_entity,
                    related_work=work,
                    start_date_written=row["date-combined"],
                    end_date_written=row["date-combined"],
                    relation_type=person_work_rel_type
                )
            
            